# slot_vsibench_check:只靠物体槽,VSI-Bench 能答对多少

**问题**:如果把「物体槽」接进 predictor,而且 predictor 能完全读懂它,题目能答到什么程度?这里不训练任何东西,直接用槽的位置按题目的规则算答案,和标准答案比。

**做法**(脚本 `jepa3d/tools/slots_answer_vsibench.py`)
1. 从题目里解析出物体名(正则,方向题 3 个名字,相对距离题 4 个选项 + 1 个参照物)。
2. 每个名字:dino.txt 在这道题的 8 张图里选一个 patch(centered 打分),点提示头把它抬成 3D 位置(冻结的 3DRAE token)。
3. 按题目规则算答案:
   - 方向题:站在 A、面向 B,C 在哪边。易:左 / 右;中:左 / 右 / 后(转 ≥ 135° 算后);难:四个象限。
   - 相对距离题:哪个选项的位置离参照物最近(题目按最近点量,这里用槽的位置,即近似物体中心)。
   - 出现顺序题:只用 dino.txt。每个名字取「该视角最高分 ≥ 它最好视角的某个比例」的第一个视角,选与之最一致的选项。

**数据**:VSI-Bench 带位姿的 8 帧版本(`vsibench_test_posed_f8`,1845 题,ScanNet + ScanNet++;去偏子集 811 题)。路径规划题不处理。帧和 hardneg_v1 模型评测时用的完全相同。

**对照**
- `随机 patch`:每个名字随机取一个 patch 再抬成 3D,等于盲猜。
- `hardneg_v1 模型`:目前的 VL-JEPA(`experiments/hardneg_v1`,同一份题和帧)。它的 real − shuffled 只有 +0.5,即基本没用图像。

**注意**:这些场景没有可用的 3D 框(EmbodiedScan 把 ScanNet 验证集放在无标注的测试划分里),所以这里只能和题目答案比,不能检查槽本身指没指中。点提示头只在 ScanNet 上训练过,ScanNet++ 对它是新数据集。

In [1]:
import json
import os
import numpy as np
import pandas as pd
pd.set_option("display.width", 220)

def load(path):
    return json.load(open(path)) if os.path.exists(path) else None

def pct(m, key):
    return f'{100 * m[key]["value"]:.1f}' if key in m else ""

def pct_ci(m, key):
    return f'{100 * m[key]["value"]:.1f} [{100 * m[key]["ci95"][0]:.1f}, {100 * m[key]["ci95"][1]:.1f}]' if key in m else ""

def metres(m, key, ci=False):
    if key not in m:
        return ""
    v = m[key]
    return f'{v["value"]:.2f} [{v["ci95"][0]:.2f}, {v["ci95"][1]:.2f}]' if ci else f'{v["value"]:.2f}'

R = "../results/"
MODEL = {"test": {"object_rel_direction_easy": 50.8, "object_rel_direction_medium": 29.6, "object_rel_direction_hard": 27.3,
                  "object_rel_distance": 26.2, "obj_appearance_order": 40.9},
         "debiased": {"object_rel_direction_easy": 51.3, "object_rel_direction_medium": 45.9, "object_rel_direction_hard": 23.9,
                      "object_rel_distance": 25.8, "obj_appearance_order": 36.2}}
TYPES = [("object_rel_direction_easy", "方向(易,2 选 1)"), ("object_rel_direction_medium", "方向(中,3 选 1)"),
         ("object_rel_direction_hard", "方向(难,4 选 1)"), ("object_rel_distance", "相对距离(4 选 1)")]
runs = {(probe, split): load(f"{R}{probe}_{split}.json") for probe in ("centre", "box05") for split in ("test", "debiased")}
runs = {k: v["metrics"] for k, v in runs.items() if v}

def table(probe, split, subset="all", sources=("argmax", "smooth3", "consensus")):
    m = runs[probe, split]
    rows = []
    for t, label in TYPES:
        if f"{subset}/{t}/argmax_acc" not in m:
            continue
        rows.append({"题型": label, "题数": m[f"{subset}/{t}/argmax_acc"]["n"], "随机 patch": pct(m, f"{subset}/{t}/random_acc"),
                     "hardneg_v1 模型": MODEL[split][t] if subset == "all" else "",
                     **{f"槽:{s}": pct_ci(m, f"{subset}/{t}/{s}_acc") for s in sources}})
    return pd.DataFrame(rows)

## 1. 测试集(1845 题里可解析的方向题和相对距离题)

`槽:argmax` 是串联测试的原做法;`smooth3`、`consensus` 是 `teacher_hit_rate` 里的另两种选点方式。点提示头:原来的(只用中心点训练)。方括号是按场景重采样的 95% 置信区间。

In [2]:
table("centre", "test")

,题型,题数,随机 patch,hardneg_v1 模型,槽:argmax,槽:smooth3,槽:consensus
0,"方向(易,2 选 1)",120,45.8,50.8,"77.5 [69.1, 86.0]","76.7 [68.3, 84.9]","85.8 [79.2, 92.6]"
1,"方向(中,3 选 1)",243,43.6,29.6,"70.4 [61.9, 79.1]","70.0 [63.1, 76.3]","65.4 [58.8, 72.5]"
2,"方向(难,4 选 1)",231,25.1,27.3,"64.9 [57.1, 71.7]","67.1 [60.2, 73.8]","57.6 [50.2, 65.7]"
3,相对距离(4 选 1),543,27.6,26.2,"44.4 [39.3, 49.2]","41.8 [37.2, 47.6]","39.8 [33.5, 45.0]"


## 2. 去偏子集

In [3]:
table("centre", "debiased")

,题型,题数,随机 patch,hardneg_v1 模型,槽:argmax,槽:smooth3,槽:consensus
0,"方向(易,2 选 1)",117,49.6,51.3,"76.9 [68.9, 85.5]","76.1 [68.1, 84.7]","85.5 [78.8, 92.3]"
1,"方向(中,3 选 1)",37,43.2,45.9,"51.4 [33.3, 69.1]","56.8 [38.5, 75.1]","45.9 [30.7, 63.4]"
2,"方向(难,4 选 1)",67,28.4,23.9,"49.3 [33.3, 63.2]","52.2 [40.0, 67.1]","47.8 [32.4, 61.3]"
3,相对距离(4 选 1),209,27.3,25.8,"49.8 [41.5, 57.3]","51.2 [44.3, 58.8]","43.5 [34.7, 51.4]"


## 3. 按数据集拆分(测试集)

点提示头只在 ScanNet 上训练过。

In [4]:
pd.concat([table("centre", "test", s).assign(数据集=s) for s in ("scannet", "scannetpp")]).drop(columns="hardneg_v1 模型")

,题型,题数,随机 patch,槽:argmax,槽:smooth3,槽:consensus,数据集
0,"方向(易,2 选 1)",63,54.0,"74.6 [63.5, 87.3]","77.8 [66.7, 88.9]","87.3 [77.8, 95.2]",scannet
1,"方向(中,3 选 1)",147,47.6,"79.6 [70.5, 88.4]","78.2 [69.2, 86.8]","73.5 [63.8, 80.9]",scannet
2,"方向(难,4 选 1)",133,23.3,"66.2 [58.6, 75.3]","70.7 [61.8, 78.3]","60.9 [51.5, 69.8]",scannet
3,相对距离(4 选 1),259,27.0,"49.0 [40.0, 58.5]","49.4 [40.9, 59.2]","44.0 [33.9, 53.4]",scannet
0,"方向(易,2 选 1)",57,36.8,"80.7 [68.4, 92.9]","75.4 [58.2, 90.9]","84.2 [73.3, 94.4]",scannetpp
1,"方向(中,3 选 1)",96,37.5,"56.2 [44.3, 69.9]","57.3 [47.3, 68.8]","53.1 [41.4, 64.9]",scannetpp
2,"方向(难,4 选 1)",98,27.6,"63.3 [52.0, 75.8]","62.2 [51.0, 75.8]","53.1 [39.0, 66.4]",scannetpp
3,相对距离(4 选 1),284,28.2,"40.1 [35.1, 45.8]","34.9 [29.8, 39.9]","35.9 [29.4, 41.6]",scannetpp


## 4. 换成偏心点重训的点提示头(`point_probe_offcentre` 的 box05)

In [5]:
pd.concat([table("box05", s).assign(划分=s) for s in ("test", "debiased")]) if ("box05", "test") in runs else print("没有 box05 的结果")

,题型,题数,随机 patch,hardneg_v1 模型,槽:argmax,槽:smooth3,槽:consensus,划分
0,"方向(易,2 选 1)",120,49.2,50.8,"75.8 [67.2, 84.9]","75.0 [66.9, 84.0]","80.0 [71.7, 88.5]",test
1,"方向(中,3 选 1)",243,44.0,29.6,"74.1 [67.7, 81.4]","72.0 [65.1, 78.9]","67.9 [59.2, 74.7]",test
2,"方向(难,4 选 1)",231,26.8,27.3,"67.1 [59.6, 74.2]","66.2 [58.8, 73.5]","61.5 [55.3, 68.2]",test
3,相对距离(4 选 1),543,28.4,26.2,"44.8 [39.0, 51.0]","43.1 [38.0, 49.2]","44.2 [37.2, 49.7]",test
0,"方向(易,2 选 1)",117,53.8,51.3,"75.2 [66.4, 84.2]","74.4 [64.0, 82.6]","79.5 [70.1, 86.6]",debiased
1,"方向(中,3 选 1)",37,37.8,45.9,"70.3 [50.0, 86.5]","59.5 [38.5, 78.4]","45.9 [30.0, 63.6]",debiased
2,"方向(难,4 选 1)",67,29.9,23.9,"50.7 [33.8, 62.7]","50.7 [37.5, 64.2]","47.8 [34.8, 60.3]",debiased
3,相对距离(4 选 1),209,25.8,25.8,"53.6 [45.3, 61.7]","54.1 [47.0, 61.8]","48.3 [38.9, 57.4]",debiased


## 5. dino.txt 的把握和答对率

每道题取各物体「最高分 patch 的标准化分数」里的最小值,即最没把握的那个物体。按这个值把题分成三等份,看 `smooth3` 的准确率(测试集)。

In [6]:
rows = [json.loads(line) for line in open(R + "centre_test.questions.jsonl")]
out = []
for t, label in TYPES:
    r = [x for x in rows if x["type"] == t]
    conf = np.array([min(x["peak_z"]) for x in r]); ok = np.array([x["smooth3"][0] == x["target"].lower() for x in r])
    lo, hi = np.quantile(conf, [1 / 3, 2 / 3])
    out.append({"题型": label, "把握最低的 1/3": f"{100 * ok[conf < lo].mean():.1f}", "中间 1/3": f"{100 * ok[(conf >= lo) & (conf < hi)].mean():.1f}",
                "把握最高的 1/3": f"{100 * ok[conf >= hi].mean():.1f}"})
pd.DataFrame(out)

,题型,把握最低的 1/3,中间 1/3,把握最高的 1/3
0,"方向(易,2 选 1)",60.5,70.7,97.6
1,"方向(中,3 选 1)",59.3,69.1,81.5
2,"方向(难,4 选 1)",59.7,63.6,77.9
3,相对距离(4 选 1),32.4,44.0,48.9


## 6. 出现顺序题:只用 dino.txt 的逐视角分数

列是「比例」阈值:该视角最高分达到这个名字最好视角的多少,就算它在这一帧出现了。1.0 = 直接取最高分的那一帧。4 选 1,随机 25%。

In [7]:
out = []
for split in ("test", "debiased"):
    m = runs["centre", split]
    for subset in ("all", "scannet", "scannetpp"):
        out.append({"划分": split, "数据集": subset, "题数": m[f"{subset}/obj_appearance_order/order0.8_acc"]["n"],
                    "hardneg_v1 模型": MODEL[split]["obj_appearance_order"] if subset == "all" else "",
                    **{str(s): pct(m, f"{subset}/obj_appearance_order/order{s}_acc") for s in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0)}})
pd.DataFrame(out)

,划分,数据集,题数,hardneg_v1 模型,0.5,0.6,0.7,0.8,0.9,1.0
0,test,all,618,40.9,31.4,33.8,33.7,31.4,30.9,30.1
1,test,scannet,390,,27.4,30.3,30.8,27.9,30.0,30.3
2,test,scannetpp,228,,38.2,39.9,38.6,37.3,32.5,29.8
3,debiased,all,318,36.2,33.0,35.8,34.3,31.8,29.9,28.9
4,debiased,scannet,180,,26.1,30.6,31.1,28.3,28.3,30.0
5,debiased,scannetpp,138,,42.0,42.8,38.4,36.2,31.9,27.5


## 结论

- **方向题上,物体槽直接算出的答案比 hardneg_v1 模型高 27–41 个点**(测试集:易 77.5 对 50.8,中 70.4 对 29.6,难 64.9 对 27.3)。模型在这些题上接近随机,而答案所需的信息在槽里。
- **相对距离题提升较小**(44.4 对 26.2):每题 5 个物体都要指对,而且题目按物体最近点量距离,槽给的是中心。
- **出现顺序题不行**(30–34%,模型 40.9):dino.txt 的逐帧分数判断不了物体在哪一帧出现,这类题不该靠槽。
- **dino.txt 的把握和答对率相关**:方向易档,把握最高的三分之一 97.6%,最低的三分之一 60.5%。槽可以带一个置信度。
- **ScanNet++ 上低一些**(方向中档 56.2 对 79.6,距离 40.1 对 49.0),但仍高于随机。
- **选点方法之间没有稳定的优劣**:consensus 在易档最高(85.8)、在中档和难档最低,差别大多在置信区间内。
- **换成 box05 点提示头,差别在置信区间内**(测试集方向中档 74.1 对 70.4,难档 67.1 对 64.9,易档 75.8 对 77.5)。

**局限**

- 这些场景没有可用的 3D 框,只能和题目答案比,不能检查槽本身。
- 这是「predictor 完美读取槽」时的分数,不是训练后能达到的分数。
- 题数少:方向易档 120 题,置信区间约 ±8 个点;去偏子集的方向中档只有 37 题。
- 物体名用正则从题目里取,只覆盖方向、相对距离、出现顺序三类题。